# Week 2 — Operational Data Cleaner & Insight Generator
### Fuel Processing Plant — Sensor Log Analysis

---
**Dataset:** `ops_sensor_log_dirty.csv`
**Primary metric:** `pressure_bar` — pipeline pressure in bar  
**Objective:** Build a reusable cleaning pipeline, perform time-series analysis, and surface hidden operational patterns invisible in the raw data.


#### 0. Environment Setup

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# Optional — visual null heatmap
try:
    import missingno as msno
    HAS_MISSINGNO = True
except ImportError:
    HAS_MISSINGNO = False
    print("missingno not installed — pip install missingno to enable visual null map")

print("Libraries loaded ✓")


missingno not installed — pip install missingno to enable visual null map
Libraries loaded ✓


### 1. Ingestion & Data Health Report

loading the raw CSV and running three diagnostic tools:
- `.info()` — column types and null counts at a glance
- `.describe()` — statistical distribution of each numeric column
- `missingno` — visual pattern of where nulls cluster (if installed)


In [17]:
df_raw = pd.read_csv("ops_sensor_log_dirty.csv")

print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print()
df_raw.info()


Shape: 1,018 rows × 6 columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1018 entries, 0 to 1017
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   timestamp      1018 non-null   object 
 1   pressure_bar   980 non-null    float64
 2   temperature_c  988 non-null    float64
 3   flow_rate_m3h  993 non-null    float64
 4   zone           1018 non-null   object 
 5   operator_id    1018 non-null   object 
dtypes: float64(3), object(3)
memory usage: 47.8+ KB


In [ ]:
df_raw.describe().round(3)


,pressure_bar,temperature_c,flow_rate_m3h
count,980.000,988.000,993.000
mean,5.822,71.907,320.569
std,11.443,5.793,30.760
min,-9.900,60.560,251.840
25%,3.993,66.376,294.450
50%,4.220,72.083,321.082
75%,4.522,77.474,347.785
max,99.500,82.940,390.928


In [ ]:
if HAS_MISSINGNO:
    fig, ax = plt.subplots(figsize=(10, 4))
    msno.matrix(df_raw, ax=ax, sparkline=False, fontsize=9)
    ax.set_title("Missing Value Map — ops_sensor_log_dirty.csv", fontsize=11, pad=10)
    plt.tight_layout()
    plt.show()
else:
    print("Null counts per column:")
    print(df_raw.isnull().sum())


Null counts per column:
timestamp         0
pressure_bar     38
temperature_c    30
flow_rate_m3h    25
zone              0
operator_id       0
dtype: int64


###  Data Health Report

After profiling the raw dataset (**1,018 rows × 6 columns**), I identified **five specific data quality issues**:

---

**Issue 1 — Unparseable Timestamps (literal string `"NaT"`)**  
The `timestamp` column arrives as plain text. Six rows contain the literal string `"NaT"` rather than a valid datetime. These rows have no recoverable position in the time series and must be dropped entirely before any time-based operation. All other timestamps parse cleanly.

---

**Issue 2 — Physically Impossible Pressure Readings (18 values)**  
`pressure_bar` ranges from **-9.9** to **99.5** in the raw data. The operating envelope for this plant is 0–15 bar. Exactly 18 readings breach this boundary (16 above 15 bar, 2 below 0). These are not operational events — they are sensor transmission faults. Left in place, they inflate the standard deviation to 11.4 and make the mean (5.82) meaningless, concealing the true signal entirely.

---

**Issue 3 — Missing Sensor Values (~9% of cells)**  
`pressure_bar` has 38 nulls, `temperature_c` has 30, `flow_rate_m3h` has 25. Because this is a continuous 10-minute time series, **dropping** these rows would create irregular time gaps that distort resampling and rolling calculations. **Linear interpolation** is the correct fix — it estimates each missing value from its immediate neighbours, which is physically justified for sensor readings that change gradually.

---

**Issue 4 — Duplicate Rows (10 exact duplicates)**  
Ten rows are exact copies of earlier readings, arising from data pipeline retransmission errors. Keeping them would double-count those time windows in any aggregation.

---